# Preparacion de los datos

In [132]:
import pandas as pd
import numpy as np

database = pd.read_csv('Breast_Cancer_Wisconsin.csv')
database = database.drop_duplicates()
x_dataset = database.drop(columns=['Unnamed: 32','id','diagnosis']).dropna()
y_dataset= database['diagnosis'].map({'B': 0, 'M': 1})

#y_dataset.head()
#x_dataset.head()

In [133]:
print(type(y_dataset))
print(type(x_dataset))

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


In [134]:
from sklearn.model_selection import train_test_split
import torch 
from sklearn.preprocessing import StandardScaler

# Separar Train-Test 

x_train,x_test,y_train,y_test = train_test_split(
    x_dataset,y_dataset, test_size=0.2 ,train_size=0.8,random_state=1,stratify=y_dataset)


# Escalar los datos
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(y_train.dtypes)


int64


In [135]:
# combierto de array a tensor:
t_x_train = torch.from_numpy(x_train).float().to('cpu')
t_x_test = torch.from_numpy(x_test).float().to('cpu')

t_y_train = torch.from_numpy(y_train.values).float().unsqueeze(1).to('cpu')
t_y_test = torch.from_numpy(y_test.values).float().unsqueeze(1).to('cpu')


print(f"x train: {t_x_train.shape}, x test: {t_x_test.shape} \t y train: {t_y_train.shape}, y test: {t_y_test.shape}")

x train: torch.Size([455, 30]), x test: torch.Size([114, 30]) 	 y train: torch.Size([455, 1]), y test: torch.Size([114, 1])


In [136]:
# Empaquetar los tensores en Train-Test y hacer Dataloader

from torch.utils.data import DataLoader,TensorDataset

train_dataset = TensorDataset(t_x_train,t_y_train)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

test_dataset =TensorDataset(t_x_test,t_y_test)

print(train_dataset[0])


(tensor([-0.3468, -0.6832, -0.3914, -0.3960, -1.2188, -0.9594, -0.6307, -0.6512,
         0.0701, -0.8874, -0.7932, -0.5276, -0.7095, -0.5816, -0.5858, -0.5349,
        -0.2903, -0.5487, -0.2298, -0.6013, -0.4964, -0.5791, -0.4917, -0.4928,
        -1.0067, -0.6662, -0.4675, -0.4967,  0.1981, -0.8104]), tensor([0.]))


# Red Neuronal

In [137]:
np.random.seed(42)
torch.manual_seed(42)

print(x_dataset.shape)
print(y_dataset.shape)

n_entradas = x_dataset.shape[1]

(569, 30)
(569,)


In [138]:
import torch.nn as nn

class Red(nn.Module):

    def __init__(self,n_entradas):
        super().__init__()

        self.capa1 = nn.Linear(n_entradas,20)
        self.capa2 = nn.Linear(20,10)
        self.capa3 = nn.Linear(10,1)

    def forward(self,inputs):
        pred_1 = torch.relu(self.capa1(inputs))
        pred_2 = self.capa2(pred_1)
        pred_final = torch.sigmoid(self.capa3(pred_2))

        return pred_final

In [139]:
lr = 0.01
epochs = 100
estatus_print = 50

In [140]:
import torch.optim as optim

model = Red(n_entradas=n_entradas)
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=lr)

# Entrenamiento

In [141]:
historico = []

for epoch in range(1,epochs+1):

    model.train()
    
    for x_batch,y_batch in train_loader:

        optimizer.zero_grad()

        y_pred = model(x_batch)
        loss = loss_fn(y_pred,y_batch)
        loss.backward()
        optimizer.step()


    model.eval()
    #  TESTING.
    with torch.no_grad():
        # Accuracy test
        y_pred_test = model(t_x_test)
        y_pred_class_test = (y_pred_test >= 0.5).float()
        accuracy_test = ((y_pred_class_test == t_y_test).float().mean()*100).round()

        # Accuracy train
        y_pred_train = model(t_x_train)
        y_pred_class_train = (y_pred_train >= 0.5).float()
        accuracy_train = ((y_pred_class_train == t_y_train).float().mean()*100).round()
        

    if epoch % estatus_print == 0:
        print(f"Epoch: {epoch} \t Loss: {loss.item():.6f}")
        print(f"Accuracy TEST: {accuracy_test} \t Accuracy TRAIN: {accuracy_train}")


Epoch: 50 	 Loss: 0.000000
Accuracy TEST: 98.0 	 Accuracy TRAIN: 100.0
Epoch: 100 	 Loss: 0.000001
Accuracy TEST: 97.0 	 Accuracy TRAIN: 100.0
